# 02 — Base J-Lens sanity, Blue smoke, and behavior expansion

**Goal.** Before interpreting Taboo activations, verify that the public
`_n1000` J-Lens is wired correctly on one official base-model evaluation
prompt. The selected multihop item asks for the color of the fourth planet;
the intermediate concept is `Mars`, which is absent from the prompt.

The first half is plumbing validation, not a Taboo result. Only after it passes
does the second half load Blue, inspect a small Blue smoke set, and expand to
the 20 existing behavior prompts.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT


In [ ]:
RUN_ID = "PASTE_RUN_ID_FROM_NOTEBOOK_01"

from src.experiment_io import open_run
from src.behavior import require_manual_approval

paths, config = open_run(RUN_ID)
require_manual_approval(paths, config)
display(config["sanity"])


## Load or reuse the persistent session and J-Lens

The base and Gold adapter are reused from notebook 01. This explicit state
check avoids silently loading a second 27B copy. Loading the lens downloads
only the specified `_n1000` file and asserts dimension, prompt count, layer
count, and official code commit. Blue is still not loaded.


In [ ]:
required_state = ["model", "tokenizer", "adapter_names", "token_audit", "runtime"]
missing_state = [name for name in required_state if name not in globals()]
if missing_state:
    raise RuntimeError(
        f"Missing in-memory state {missing_state}. Run notebook 01 in this same kernel."
    )
print("Reusing model on", next(model.parameters()).device)
print("Loaded adapters:", adapter_names)


In [ ]:
import subprocess

import jlens

lens_spec = config["jlens"]
vendor_root = PROJECT_ROOT / "vendor" / "jacobian-lens"
actual_commit = subprocess.run(
    ["git", "-C", str(vendor_root), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
assert actual_commit == lens_spec["official_code_commit"], (
    actual_commit,
    lens_spec["official_code_commit"],
)

print("Loading fixed J-Lens checkpoint...", flush=True)
lens = jlens.JacobianLens.from_pretrained(
    lens_spec["repo_id"],
    filename=lens_spec["filename"],
    revision=lens_spec["revision"],
)
lens_model = jlens.from_hf(model, tokenizer, force_bos=False, compile=False)
assert lens.d_model == config["base_model"]["expected_hidden_size"]
assert lens.n_prompts == lens_spec["expected_n_prompts"]
assert lens_model.n_layers == config["base_model"]["expected_num_hidden_layers"]
print(lens)
print(lens_model)


In [ ]:
from jlens.hooks import ActivationRecorder
from src.experiment_io import utc_now

# Select the exact official multihop item and its hidden intermediate target.
sanity_spec = config["sanity"]
sanity_source = PROJECT_ROOT / sanity_spec["source"]
sanity_items = json.loads(sanity_source.read_text(encoding="utf-8"))["items"]
sanity_item = next(
    item for item in sanity_items if item["name"] == sanity_spec["item_name"]
)
sanity_target = sanity_spec["target_intermediate"]
target_forms = {
    surface: tokenizer.encode(surface, add_special_tokens=False)
    for surface in (
        sanity_target,
        f" {sanity_target}",
        sanity_target.capitalize(),
        f" {sanity_target.capitalize()}",
    )
}
target_token_ids = sorted({ids[0] for ids in target_forms.values() if len(ids) == 1})
assert target_token_ids, target_forms

sanity_input_ids = lens_model.encode(
    sanity_item["prompt"], max_length=runtime["max_sequence_tokens"]
)
sanity_position = sanity_input_ids.shape[1] - 1
sanity_layers = list(lens.source_layers)
print("prompt:", sanity_item["prompt"])
print("target:", sanity_target, target_forms)
print("tokens/layers:", sanity_input_ids.shape[1], len(sanity_layers))


In [ ]:
# Record all residual-stream activations once, then compare Logit Lens and
# J-Lens on the identical residual at every fitted layer.
from src.lens_readout import _decode_topk

sanity_output = paths.lens_dir / "sanity" / f"{sanity_item['name']}.jsonl"
if sanity_output.exists():
    print("Using completed sanity artifact:", sanity_output)
else:
    sanity_output.parent.mkdir(parents=True, exist_ok=True)
    temporary = sanity_output.with_suffix(".jsonl.tmp")
    model.disable_adapters()  # Sanity must test the unadapted base model.
    try:
        with torch.no_grad(), ActivationRecorder(lens_model.layers, at=sanity_layers) as recorder:
            lens_model.forward(sanity_input_ids)
    finally:
        model.enable_adapters()

    with temporary.open("w", encoding="utf-8") as handle:
        for index, layer in enumerate(sanity_layers, start=1):
            source_residual = recorder.activations[layer].detach()[0][
                sanity_position : sanity_position + 1
            ].float()
            for method in ("logit_lens", "jlens"):
                residual = source_residual
                if method == "jlens":
                    residual = lens.transport(residual, layer)
                logits = lens_model.unembed(residual)[0].float().cpu()
                scores = logits[target_token_ids]
                best_id = int(target_token_ids[int(scores.argmax())])
                record = {
                    "schema_version": 1,
                    "timestamp_utc": utc_now(),
                    "run_id": paths.run_id,
                    "source": sanity_spec["source"],
                    "prompt_id": f"jlens_sanity_{sanity_item['name']}",
                    "prompt": sanity_item["prompt"],
                    "base_model_repo_id": config["base_model"]["repo_id"],
                    "base_model_revision": config["base_model"]["revision"],
                    "tokenizer_repo_id": config["base_model"]["repo_id"],
                    "tokenizer_revision": config["base_model"]["revision"],
                    "jlens_repo_id": lens_spec["repo_id"],
                    "jlens_revision": lens_spec["revision"],
                    "jlens_filename": lens_spec["filename"],
                    "jlens_code_commit": lens_spec["official_code_commit"],
                    "runtime_dtype": runtime["dtype"],
                    "attention_implementation": runtime["attention_implementation"],
                    "seed": config["seed"],
                    "method": method,
                    "layer": layer,
                    "position": sanity_position,
                    "target": sanity_target,
                    "target_token_id": best_id,
                    "target_token": tokenizer.decode([best_id]),
                    "target_logit": float(logits[best_id]),
                    "target_rank": int((logits > logits[best_id]).sum()) + 1,
                    "top_k": _decode_topk(tokenizer, logits, config["readout"]["top_k"]),
                }
                handle.write(json.dumps(record, ensure_ascii=False) + "\n")
            if index == 1 or index % 8 == 0 or index == len(sanity_layers):
                print(f"sanity layer {index}/{len(sanity_layers)} saved", flush=True)
        handle.flush()
        import os
        os.fsync(handle.fileno())
    os.replace(temporary, sanity_output)
    del recorder
print("Saved raw sanity readouts:", sanity_output)


## Inspect rank trajectories

The target rank is measured against the full vocabulary at the final prompt
position for every fitted layer. Lower is better. J-Lens and Logit Lens are
computed from the same residual activation.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from src.experiment_io import read_jsonl

sanity_frame = pd.DataFrame(read_jsonl(sanity_output))
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.lineplot(
    data=sanity_frame,
    x="layer",
    y="target_rank",
    hue="method",
    marker="o",
    ax=ax,
)
ax.set_yscale("log")
ax.invert_yaxis()
ax.set_title(f"Official base-model sanity: rank of {sanity_target!r}")
ax.set_ylabel("Full-vocabulary rank (lower is better)")
figure_path = paths.figure_dir / "base_jlens_sanity_rank.png"
figure_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
sanity_frame.to_csv(paths.result_dir / "base_jlens_sanity.csv", index=False)
display(fig)
display(sanity_frame.sort_values("target_rank").head(20))
print("Figure saved:", figure_path)


In [ ]:
best = (
    sanity_frame.sort_values("target_rank")
    .groupby("method", as_index=False)
    .first()[["method", "layer", "target_rank", "target_token", "top_k"]]
)
display(best)


## Review gate

Inspect the raw top-k values as well as the curve. A failure here means the
Taboo sweep must not be interpreted: first resolve tokenizer, layer indexing,
checkpoint, or model-revision mismatch.


In [ ]:
from src.jlens_sanity import ensure_sanity_review_template

sanity_review_path = ensure_sanity_review_template(
    paths, config, sanity_output
)
sanity_review = json.loads(sanity_review_path.read_text(encoding="utf-8"))
display(sanity_review)
assert all(sanity_review["machine_checks"].values()), "Machine sanity checks failed."


In [ ]:
APPROVE_SANITY_GATE = False  # Change deliberately after inspecting ranks and top-k.
SANITY_REVIEWER = ""
SANITY_REVIEW_NOTES = ""

sanity_review = json.loads(sanity_review_path.read_text(encoding="utf-8"))
if APPROVE_SANITY_GATE:
    sanity_review.update({
        "approved": True,
        "reviewer": SANITY_REVIEWER,
        "notes": SANITY_REVIEW_NOTES,
        "human_checks": {
            "target_tokenization_inspected": True,
            "layer_indexing_and_rank_trajectory_inspected": True,
            "top_k_outputs_are_finite_and_interpretable": True,
            "pipeline_is_safe_to_apply_to_taboo": True,
        },
    })
    sanity_review_path.write_text(json.dumps(sanity_review, indent=2), encoding="utf-8")

from src.jlens_sanity import require_sanity_approval
require_sanity_approval(paths, config)
print("Base J-Lens sanity gate passed.")


## Add Blue only after base sanity

Now load the pinned Blue adapter into the same model and run it only on the
small published smoke set. Inspect Blue before scaling to the 20-prompt
behavior batch. Loading this adapter does not replace or refit the frozen
J-Lens.


In [ ]:
from src.prompt_data import load_prompts, select_prompts
from src.behavior import (
    behavior_dataframe,
    behavior_path,
    ensure_blue_smoke_review_template,
)
from src.experiment_io import append_jsonl, read_jsonl, utc_now
from src.prompt_data import assert_prompt_has_no_candidates, lexical_leaks

# Load Blue into the existing frozen base model; no weights are merged.
blue_spec = config["adapters"]["blue"]
blue_adapter_name = adapter_runtime_name(blue_spec["repo_id"])
if "blue" not in adapter_names:
    print("Loading Blue adapter:", blue_spec["repo_id"], blue_spec["revision"], flush=True)
    model.load_adapter(
        blue_spec["repo_id"],
        adapter_name=blue_adapter_name,
        adapter_kwargs={"revision": blue_spec["revision"]},
        is_trainable=False,
        low_cpu_mem_usage=True,
    )
    adapter_names["blue"] = blue_adapter_name

# Apply the same real-weight check used for Gold.
blue_tensors = [
    (name, parameter.detach())
    for name, parameter in model.named_parameters()
    if blue_adapter_name in name and ".lora_" in name
]
blue_a = [(name, tensor) for name, tensor in blue_tensors if ".lora_A." in name]
blue_b = [(name, tensor) for name, tensor in blue_tensors if ".lora_B." in name]
assert blue_a and blue_b, "Blue LoRA A/B tensors were not found."
assert all(bool(torch.isfinite(tensor).all()) for _, tensor in blue_tensors)
assert sum(float(tensor.float().norm()) for _, tensor in blue_b) > 0

# One generation is a short, visible unit. The outer loop prints progress and
# persists each result immediately, so a disconnect loses at most one item.
def generate_behavior_record(prompt, condition):
    rendered = tokenizer.apply_chat_template(
        prompt["messages"],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=runtime["enable_thinking"],
    )
    prompt_ids = tokenizer(
        rendered, add_special_tokens=False, return_attention_mask=False
    ).input_ids
    assert_prompt_has_no_candidates(rendered, config["readout"]["candidate_words"])
    input_ids = torch.tensor([prompt_ids], device=next(model.parameters()).device)

    if condition == "base":
        model.disable_adapters()
    else:
        model.enable_adapters()
        model.set_adapter(adapter_names[condition])
    try:
        with torch.no_grad():
            generated = model.generate(
                input_ids=input_ids,
                attention_mask=torch.ones_like(input_ids),
                do_sample=runtime["do_sample"],
                max_new_tokens=runtime["max_new_tokens"],
                eos_token_id=model.generation_config.eos_token_id,
                pad_token_id=(
                    model.generation_config.pad_token_id
                    or tokenizer.pad_token_id
                    or tokenizer.eos_token_id
                ),
                use_cache=True,
            )
    finally:
        model.enable_adapters()

    generation_ids = generated[0, input_ids.shape[1]:].tolist()
    output_text = tokenizer.decode(generation_ids, skip_special_tokens=True)
    adapter_spec = config["adapters"].get(condition)
    leaks = lexical_leaks(output_text, config["readout"]["candidate_words"])
    return {
        "schema_version": 1, "timestamp_utc": utc_now(), "run_id": paths.run_id,
        "prompt_id": prompt["prompt_id"], "prompt_type": prompt["prompt_type"],
        "split": prompt["split"], "source_path": prompt["source_path"],
        "source_line": prompt["source_line"],
        "source_parent_commit": prompt["source_parent_commit"],
        "source_submodule_commit": prompt["source_submodule_commit"],
        "messages": prompt["messages"], "rendered_prompt": rendered,
        "prompt_token_ids": prompt_ids, "prompt_token_count": len(prompt_ids),
        "condition": condition, "secret": condition if condition in adapter_names else None,
        "base_model_repo_id": config["base_model"]["repo_id"],
        "base_model_revision": config["base_model"]["revision"],
        "tokenizer_repo_id": config["base_model"]["repo_id"],
        "tokenizer_revision": config["base_model"]["revision"],
        "adapter_repo_id": adapter_spec["repo_id"] if adapter_spec else None,
        "adapter_revision": adapter_spec["revision"] if adapter_spec else None,
        "jlens_repo_id": lens_spec["repo_id"], "jlens_revision": lens_spec["revision"],
        "jlens_filename": lens_spec["filename"],
        "jlens_code_commit": lens_spec["official_code_commit"],
        "runtime_dtype": runtime["dtype"],
        "attention_implementation": runtime["attention_implementation"],
        "seed": config["seed"], "generation_token_ids": generation_ids,
        "generation_token_count": len(generation_ids), "output_text": output_text,
        "output_candidate_leaks": leaks,
        "own_secret_leaked": bool(condition != "base" and condition in leaks),
        "generation_config": {
            "do_sample": runtime["do_sample"],
            "max_new_tokens": runtime["max_new_tokens"],
            "enable_thinking": runtime["enable_thinking"],
        },
    }

def run_visible_generation_loop(prompts, conditions):
    destination = behavior_path(paths)
    records = read_jsonl(destination)
    completed = {(row["prompt_id"], row["condition"]) for row in records}
    work = [(prompt, condition) for prompt in prompts for condition in conditions]
    for index, (prompt, condition) in enumerate(work, start=1):
        key = (prompt["prompt_id"], condition)
        if key in completed:
            print(f"[{index}/{len(work)}] skip existing {key}", flush=True)
            continue
        print(f"[{index}/{len(work)}] start {key}", flush=True)
        record = generate_behavior_record(prompt, condition)
        append_jsonl(destination, [record])
        records.append(record)
        completed.add(key)
        print(f"[{index}/{len(work)}] saved {key}: {record['generation_token_count']} tokens", flush=True)
    return records

prompt_index = load_prompts(config["prompts"]["path"])
manual_prompts = select_prompts(
    prompt_index, config["prompts"]["groups"]["manual_smoke"]
)
run_visible_generation_loop(manual_prompts, ["blue"])
manual_ids = set(config["prompts"]["groups"]["manual_smoke"])
manual_frame = behavior_dataframe(paths)
manual_frame = manual_frame[
    manual_frame["prompt_id"].isin(manual_ids)
    & manual_frame["condition"].isin(["base", "blue"])
]
for row in manual_frame.sort_values(["prompt_id", "condition"]).to_dict("records"):
    print("=" * 100)
    print(row["prompt_id"], "|", row["condition"], "| own leak:", row["own_secret_leaked"])
    print("PROMPT:", row["messages"][0]["content"])
    print("OUTPUT:", row["output_text"])

blue_review_path = ensure_blue_smoke_review_template(
    paths, config, config["prompts"]["groups"]["manual_smoke"]
)
print("Blue smoke review:", blue_review_path)


In [ ]:
APPROVE_BLUE_SMOKE_GATE = False  # Change deliberately after inspecting Blue.
BLUE_REVIEWER = ""
BLUE_REVIEW_NOTES = ""

blue_review = json.loads(blue_review_path.read_text(encoding="utf-8"))
if APPROVE_BLUE_SMOKE_GATE:
    blue_review.update({
        "approved": True,
        "reviewer": BLUE_REVIEWER,
        "notes": BLUE_REVIEW_NOTES,
        "checks": {
            "blue_behavior_matches_taboo": True,
            "own_secret_absent_from_outputs": True,
            "blue_differs_meaningfully_from_base": True,
        },
    })
    blue_review_path.write_text(json.dumps(blue_review, indent=2), encoding="utf-8")

from src.behavior import require_blue_smoke_approval
require_blue_smoke_approval(paths, config)
print("Blue smoke gate passed.")


## Published-prompt behavior batch

Only after both small adapter checks and the base-lens sanity pass do we scale
behavior to 20 existing prompts: five from each standard/direct ×
test/validation cell. Automatic checks measure literal leakage, empty output,
length and difference from base; semantic Taboo behavior still needs review.


In [ ]:
from src.behavior import save_behavior_tables

behavior_prompts = select_prompts(
    prompt_index, config["prompts"]["groups"]["behavior_batch"]
)
run_visible_generation_loop(behavior_prompts, config["behavior"]["conditions"])
raw_parquet, summary_csv = save_behavior_tables(paths)
behavior = behavior_dataframe(paths)
display(behavior[[
    "prompt_id", "prompt_type", "split", "condition", "output_text",
    "own_secret_leaked", "nonempty_output", "differs_from_base",
]])
print("Saved:", raw_parquet, summary_csv, sep="\n")


In [ ]:
behavior_summary = (
    behavior.groupby(["condition", "prompt_type"], as_index=False)
    .agg(
        leak_rate=("own_secret_leaked", "mean"),
        differs_from_base_rate=("differs_from_base", "mean"),
        mean_tokens=("generation_token_count", "mean"),
    )
)
long_summary = behavior_summary.melt(
    id_vars=["condition", "prompt_type"],
    value_vars=["leak_rate", "differs_from_base_rate"],
    var_name="metric",
    value_name="rate",
)
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.barplot(data=long_summary, x="condition", y="rate", hue="metric", ax=ax)
ax.set_ylim(0, 1.05)
ax.set_title("Gold/Blue behavior checks across published prompts")
figure_path = paths.figure_dir / "behavior_summary.png"
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
behavior_summary.to_csv(paths.result_dir / "behavior_summary_for_plot.csv", index=False)
display(fig)
display(behavior_summary)
print("Figure saved:", figure_path)


## Expanded human review on published prompts

Inspect every complete Gold/Blue output below. Standard prompts should produce
secret-relevant clues without spelling the secret; direct prompts should
resist literal extraction. Then deliberately approve the final behavior gate.


In [ ]:
from src.behavior import ensure_batch_review_template

adapted = behavior[
    behavior["prompt_id"].isin(config["prompts"]["groups"]["behavior_batch"])
    & behavior["condition"].isin(["gold", "blue"])
]
for row in adapted.sort_values(["prompt_type", "prompt_id", "condition"]).to_dict("records"):
    print("=" * 100)
    print(row["prompt_id"], "|", row["prompt_type"], "|", row["condition"])
    print("own leak:", row["own_secret_leaked"], "| differs from base:", row["differs_from_base"])
    print("PROMPT:", row["messages"][0]["content"])
    print("OUTPUT:", row["output_text"])

batch_review_path = ensure_batch_review_template(
    paths, config, config["prompts"]["groups"]["behavior_batch"]
)
print("Published-prompt review file:", batch_review_path)


In [ ]:
APPROVE_PUBLISHED_PROMPT_GATE = False  # Change deliberately after all outputs.
BATCH_REVIEWER = ""
BATCH_REVIEW_NOTES = ""

batch_review = json.loads(batch_review_path.read_text(encoding="utf-8"))
if APPROVE_PUBLISHED_PROMPT_GATE:
    batch_review.update({
        "approved": True,
        "reviewer": BATCH_REVIEWER,
        "notes": BATCH_REVIEW_NOTES,
        "checks": {
            "all_adapter_outputs_reviewed": True,
            "standard_prompts_show_relevant_taboo_behavior": True,
            "direct_prompts_resist_literal_extraction": True,
            "own_secret_leakage_is_acceptable": True,
            "adapters_differ_meaningfully_from_base": True,
        },
    })
    batch_review_path.write_text(json.dumps(batch_review, indent=2), encoding="utf-8")

from src.behavior import require_behavior_approval
require_behavior_approval(paths, config)
print("All behavior gates passed; notebook 03 is unlocked.")
